![databricks_academy_logo.png](../Includes/images/databricks_academy_logo.png "databricks_academy_logo.png")

# Ingest and Manipulate a Delta Table Lab

This notebook provides a hands-on review of some of the features Delta Lake brings to the data lakehouse.

## Important: Select Environment 4
The cells below may not work in other environments. To choose environment 4: 
1. Click the ![environment.png](../Includes/images/environment.png "environment.png") button on the right sidebar
1. Open the **Environment version** dropdown
1. Select **4**

## Classroom Setup

Run the following cell to configure your working environment for this course.

In [0]:
####################################################################################
# Set python variables for catalog, schema, and volume names (change, if desired)
catalog_name = "dbacademy"
schema_name = "ingestion_lab"
volume_name = "myfiles"
####################################################################################

####################################################################################
# Create the catalog, schema, and volume if they don't exist already
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
####################################################################################

####################################################################################
# Creates a file called employees.csv in the specified catalog.schema.volume
import pandas as pd
data = [
    ["1111", "Kristi", "USA", "Manager"],
    ["2222", "Sophia", "Greece", "Developer"],
    ["3333", "Peter", "USA", "Developer"],
    ["4444", "Zebi", "Pakistan", "Administrator"]
]
columns = ["ID", "Firstname", "Country", "Role"] 
df = pd.DataFrame(data, columns=columns)
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/employees.csv"
df.to_csv(file_path, index=False)
################################################################################

####################################################################################
# Creates a file called employees2.csv in the specified catalog.schema.volume
spark.sql(f'CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.taxi_files')
output_volume = f'/Volumes/{catalog_name}/{schema_name}/taxi_files'

sdf = spark.table("samples.nyctaxi.trips")

(sdf
    .write
    .mode("overwrite")
    .csv(output_volume, header=True)
    )
####################################################################################

## Begin Lab

1. Set your current Catalog to **dbacademy** and your schema to **ingesting_data**.

    **HINT**:
    - Catalog: `USE CATALOG`
    - Schema: `USE SCHEMA`

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
USE CATALOG dbacademy;
USE SCHEMA ingestion_lab;

2. Run a query to view your current catalog and schema. Verify that the results show the **dbacademy** catalog and the **ingestion_lab** schema.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SELECT current_catalog(), current_schema()

3. View the available volumes in your schema and confirm that the **taxi_files** volume is listed.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SHOW VOLUMES;

4. List the files in the **taxi_files** volume and check the **name**  column to determine the file types stored in the volume. Ignore any additional files that begin with an underscore (_).

**HINT**: Use the following path format to access the volume: */Volumes/catalog_name/schema_name/volume_name/*.

In [0]:
# TODO
# <FILL_IN>

In [0]:
%skip
spark.sql(f"LIST '/Volumes/{catalog_name}/{schema_name}/taxi_files'").display()

5. Query the volume path directly and preview the data in the file using the appropriate file format. Make sure to use backticks around the path to your volume.

**HINT**: SELECT * FROM \<file-format\>.\`\<path-to-volume-taxi_files\>\`

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
# ANSWER
spark.sql(f'''SELECT *
FROM csv.`/Volumes/{catalog_name}/{schema_name}/taxi_files`
LIMIT 10
''').display()

6. Create a table in your schema called **taxitrips_bronze** that contains the following columns:
| Field Name | Field type |
| --- | --- |
| tpep_pickup_datetime | TIMESTAMP |
| tpep_dropoff_datetime | TIMESTAMP |
| trip_distance | DOUBLE |
| fare_amount | DOUBLE |
| pickup_zip | INT |
| dropoff_zip | INT |

**NOTE:** The DROP TABLE statement will drop the table if it already exists to avoid errors.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
DROP TABLE IF EXISTS taxitrips_bronze;

CREATE TABLE IF NOT EXISTS taxitrips_bronze (
  tpep_pickup_datetime TIMESTAMP,
  tpep_dropoff_datetime TIMESTAMP,
  trip_distance DOUBLE,
  fare_amount DOUBLE,
  pickup_zip INT,
  dropoff_zip INT
);

7. Use the [COPY INTO](https://docs.databricks.com/en/sql/language-manual/delta-copy-into.html) statement to populate the table with files from the **taxi_files** volume into the **taxitrips_bronze** table. Include the following options:
    - FROM `path-to-taxi_files`
    - FILEFORMAT = '\<file-format\>'
    - FORMAT_OPTIONS
      - 'header' = 'true'
      - 'inferSchema' = 'true'

    Confirm 21,932 rows were inserted.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
# ANSWER
spark.sql(f'''COPY INTO taxitrips_bronze
  FROM '/Volumes/{catalog_name}/{schema_name}/taxi_files'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
  ''').display()

8. Count the number of rows in the **taxitrips_bronze** table. Confirm that the table has 21,932 rows.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SELECT count(*) as totalrows
FROM taxitrips_bronze;

9. View the **taxitrips_bronze** table's history. Confirm version 0 and version 1 are available.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
DESCRIBE HISTORY taxitrips_bronze;

10. Run the following script to delete all rows where **trip_distance** is less than *1*. Confirm *5,387* rows were deleted.

In [0]:
%sql
DELETE FROM taxitrips_bronze
  WHERE trip_distance < 1;

11. View the **taxitrips_bronze** table's history. View the **operation** column. View the version where the *DELETE* operation occurred.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
DESCRIBE HISTORY taxitrips_bronze;

12. Run a query to count the total number of rows in the current version of the **taxitrips_bronze** table. Confirm that the current table contains *16,545* rows.

**HINT:** By default the most recent version will be used.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SELECT count(*) AS totalrows
FROM taxitrips_bronze;

13. Query the original version of the table to count the number of rows when it was first created. Confirm that the original table contains *21,932* rows.

**HINT:** FROM \<table> VERSION AS OF \<n>

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SELECT count(*) AS totalrows
FROM taxitrips_bronze VERSION AS OF 1;

**CHALLENGE**


14. Whoops! You made a mistake and didn't mean to delete the rows from earlier. Use the [RESTORE](https://docs.databricks.com/en/sql/language-manual/delta-restore.html) statement to restore a Delta table to the original state prior to the *DELETE* operation. 

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
RESTORE TABLE taxitrips_bronze TO VERSION AS OF 1;

15. View the history of the **taxitrips_bronze** table. Confirm the most recent version contains the **operation** *RESTORE*.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
DESCRIBE HISTORY taxitrips_bronze;

16. Count the total number of rows in the current **taxitrips_bronze** table. Confirm that the most recent version of the table contains *21,932* rows.

In [0]:
%sql
-- TODO
-- <FILL_IN>

In [0]:
%skip
%sql
-- ANSWER
SELECT count(*) as totalrows
FROM taxitrips_bronze;

17. Drop the **ingestion_lab** schema.

In [0]:
%sql
-- TODO
-- <FILL_IN> 

In [0]:
%skip
%sql
-- ANSWER
DROP SCHEMA IF EXISTS ingestion_lab CASCADE;

### Summary
By completing this lab, you should now feel comfortable:
* Completing standard Delta Lake table creation and data manipulation commands
* Reviewing table metadata including table history
* Leverage Delta Lake versioning for snapshot queries and rollbacks